# Faker -> Landing (SeaweedFS via Spark)

Gera dados falsos com `Faker` e grava em `s3a://landing/faker-demo/pessoas/` via Spark,
conectando no cluster Spark standalone do datastack (`spark://spark-master:7077`).

**Importante sobre este notebook (DockerSpawner):**
- Este container **não herda** o `spark-defaults.conf` do cluster (não é montado aqui) - por isso a
  SparkSession abaixo configura o S3A/magic committer manualmente.
- **O Spark deste container (`jupyter/pyspark-notebook:spark-3.5.0`) é a versão 3.5.0, mas o cluster
  (executors) roda 3.5.6.** Mesmo sendo "3.5.x", a serialização Java de tasks entre driver/executor é
  sensível a patch version (`serialVersionUID`) - conectar direto dá
  `InvalidClassException: ... local class incompatible`. Por isso a célula 3 baixa o Spark 3.5.6 exato
  (arquivo da Apache) para dentro de `work/` antes de importar o pyspark.
- O jar do magic committer não vem nesta distribuição - resolvido via `spark.jars.packages` (Maven).
- Salve este arquivo dentro de `work/` (`/home/jovyan/work/bruno.ipynb`) - é a unica pasta com volume
  persistente (`jupyterhub-user-superadmin`); qualquer coisa fora dela some se o container for reciclado
  (`DockerSpawner.remove=True`).
- **Credenciais (access key / secret key) ficam em `work/.env`, não no notebook.** Lidas via
  `python-dotenv`.

**Se você já rodou alguma célula com `import pyspark` nesta sessão antes de aplicar o fix de versão:
reinicie o kernel (Kernel > Restart Kernel) e rode tudo de novo a partir do topo** - Python não troca a
versão de um módulo já importado.

**IMPORTANTE - kernel Python:** o driver deste notebook roda *dentro* do kernel Jupyter (nao um `spark-submit` separado), entao a versao do Python do kernel precisa bater com a dos executors do cluster (Python 3.12, imagem `bitnami/spark`). Este arquivo ja esta configurado para abrir com o kernel **"Python 3.12 (bate com o cluster Spark)"** - confira no canto superior direito do JupyterLab. Se aparecer `python3` (3.11) em vez disso, troque em **Kernel > Change Kernel...**

Esse kernel foi criado com:
```bash
mamba create -y -p /home/jovyan/work/envs/py312 python=3.12 ipykernel
/home/jovyan/work/envs/py312/bin/python -m ipykernel install --user --name py312-spark \
  --display-name "Python 3.12 (bate com o cluster Spark)"
```
O ambiente conda fica em `work/envs/py312` (persistente). **O registro do kernel (`~/.local/share/jupyter/kernels/`) NAO e persistente** - se o container for reciclado e o kernel sumir da lista, so rodar o segundo comando de novo (o `ipykernel install`) para reaparecer; nao precisa recriar o ambiente inteiro.

## 1. Instalar dependências

In [ ]:
%pip install -q Faker python-dotenv

## 2. Carregar credenciais do `.env`

Le `work/.env` (mesma pasta deste notebook). Se as variáveis não existirem lá, o erro abaixo já avisa
antes de tentar conectar no Spark.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/home/jovyan/work/.env")

S3A_ACCESS_KEY = os.environ["SEAWEEDFS_ACCESS_KEY"]
S3A_SECRET_KEY = os.environ["SEAWEEDFS_SECRET_KEY"]

print("Credenciais carregadas do .env (access key: "
      f"{S3A_ACCESS_KEY[:4]}{'*' * (len(S3A_ACCESS_KEY) - 4)})")

## 3. Alinhar a versão do Spark com o cluster (3.5.6)

**Precisa rodar ANTES de qualquer `import pyspark`.** Baixa o Spark 3.5.6 (mesma versão dos executors)
direto do arquivo da Apache para `work/spark-3.5.6-bin-hadoop3` (~400MB, só baixa uma vez - fica na
pasta persistente, sobrevive a restart de container) e ajusta `sys.path` para usar essa distribuição em
vez do PySpark 3.5.0 pré-instalado na imagem.

In [ ]:
import os
import sys
import pathlib
import subprocess

SPARK_VERSION = "3.5.6"  # tem que bater EXATO com a versao do cluster (ver executors)
SPARK_HOME_CUSTOM = f"/home/jovyan/work/spark-{SPARK_VERSION}-bin-hadoop3"

if not pathlib.Path(SPARK_HOME_CUSTOM).exists():
    print(f"Baixando Spark {SPARK_VERSION} (~400MB, uma vez so)...")
    url = f"https://archive.apache.org/dist/spark/spark-{SPARK_VERSION}/spark-{SPARK_VERSION}-bin-hadoop3.tgz"
    subprocess.run(f"curl -sL {url} | tar -xz -C /home/jovyan/work/", shell=True, check=True)
    print("Extraido em", SPARK_HOME_CUSTOM)
else:
    print(f"Spark {SPARK_VERSION} ja esta em {SPARK_HOME_CUSTOM} (baixado numa execucao anterior).")

os.environ["SPARK_HOME"] = SPARK_HOME_CUSTOM
py4j_zip = next(pathlib.Path(f"{SPARK_HOME_CUSTOM}/python/lib").glob("py4j-*.zip"))
sys.path.insert(0, f"{SPARK_HOME_CUSTOM}/python")
sys.path.insert(0, str(py4j_zip))

if "pyspark" in sys.modules:
    print("\n*** ATENCAO: pyspark ja estava importado neste kernel (versao antiga)."
          " Reinicie o kernel e rode de novo a partir do topo. ***")
else:
    print(f"\nSPARK_HOME = {os.environ['SPARK_HOME']} (pronto para importar pyspark {SPARK_VERSION})")

## 4. SparkSession

Mesma receita validada no cluster (ver `CLAUDE.md` do repo `datastack-huawei`): magic committer + `directory.marker.retention=keep` (senão a escrita trava/expira em silêncio no SeaweedFS).

In [ ]:
from pyspark.sql import SparkSession
import pyspark

print(f"pyspark carregado: {pyspark.__version__} (tem que ser {SPARK_VERSION})")
assert pyspark.__version__ == SPARK_VERSION, (
    "Versao errada carregada - reinicie o kernel e rode as celulas na ordem, sem pular a celula 3."
)

spark = (
    SparkSession.builder
    .appName("faker-landing-bruno")
    .master("spark://spark-master:7077")
    # Recursos modestos - e um cluster compartilhado, standalone concede todos
    # os cores disponiveis por padrao se nao limitarmos.
    .config("spark.cores.max", "4")
    .config("spark.executor.memory", "2g")
    # Jar do magic committer (nao vem nesta distribuicao) - resolvido via Maven.
    .config("spark.jars.packages", "org.apache.spark:spark-hadoop-cloud_2.12:3.5.6")
    # Cache do Ivy na pasta persistente - evita rebaixar ~100MB+ de deps
    # transitivas (hadoop-aws/azure/gcs) toda vez que o container e reciclado.
    .config("spark.jars.ivy", "/home/jovyan/work/.ivy2")
    # ----- S3A / SeaweedFS -----
    .config("spark.hadoop.fs.s3a.endpoint", "http://seaweedfs-filer-1:8333")
    .config("spark.hadoop.fs.s3a.access.key", S3A_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", S3A_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    # SeaweedFS nao suporta bulk delete nem rename-como-copy (FileOutputCommitter).
    .config("spark.hadoop.fs.s3a.multiobjectdelete.enable", "false")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    # Sem isto, cada arquivo escrito tenta apagar marcadores de diretorio e o
    # SeaweedFS responde 500 -> retry-backoff -> escrita trava por minutos.
    .config("spark.hadoop.fs.s3a.directory.marker.retention", "keep")
    # Magic committer: escreve direto no destino via multipart, sem rename/copy.
    .config("spark.hadoop.fs.s3a.committer.name", "magic")
    .config("spark.hadoop.fs.s3a.committer.magic.enabled", "true")
    .config("spark.sql.sources.commitProtocolClass",
            "org.apache.spark.internal.io.cloud.PathOutputCommitProtocol")
    .config("spark.sql.parquet.output.committer.class",
            "org.apache.spark.internal.io.cloud.BindingParquetOutputCommitter")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} conectado em spark://spark-master:7077")
print(f"Executors: {spark.sparkContext.defaultParallelism} cores default")

## 5. Gerar dados falsos com Faker

Ajuste `N_LINHAS` como quiser. Gerado no driver (ok para dezenas de milhares de linhas; para volumes bem maiores, prefira gerar distribuído com um `flatMap` sobre um RDD particionado).

In [ ]:
from faker import Faker
from datetime import datetime, timezone
import random

fake = Faker("pt_BR")
Faker.seed(42)

N_LINHAS = 5_000

def gerar_pessoa(i):
    return {
        "id": i,
        "nome": fake.name(),
        "email": fake.email(),
        "cpf": fake.cpf(),
        "telefone": fake.phone_number(),
        "endereco": fake.street_address(),
        "cidade": fake.city(),
        "estado": fake.estado_sigla(),
        "cep": fake.postcode(),
        "data_nascimento": fake.date_of_birth(minimum_age=18, maximum_age=90).isoformat(),
        "empresa": fake.company(),
        "cargo": fake.job(),
        "salario": round(random.uniform(1800, 25000), 2),
        "_ingestion_ts": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }

rows = [gerar_pessoa(i) for i in range(N_LINHAS)]
print(f"{len(rows):,} registros gerados. Exemplo:")
rows[0]

## 6. Criar DataFrame Spark e gravar na landing

In [ ]:
LANDING_PATH = "s3a://landing/faker-demo/pessoas/"

df = spark.createDataFrame(rows)
df.printSchema()

df.write.mode("overwrite").parquet(LANDING_PATH)
print(f"Gravado em {LANDING_PATH}")

## 7. Validar (ler de volta)

In [ ]:
df_val = spark.read.parquet(LANDING_PATH)
print(f"Linhas gravadas: {df_val.count():,}")
df_val.show(10, truncate=False)

In [ ]:
spark.stop()